In [ ]:
# Read csv file and remove duplicate rows
# This part is done in aruba

scrubber = "13335"
date = "02_jan_2024"
path = "../data/02jan2024/"
import pandas as pd
df = pd.read_csv('/home/shyam/jupy/ddos_scrubber/data/optimized_raw_as'+scrubber+'_'+date+'.csv', low_memory=False)
# Remove duplicate rows in dataframe
df = df.drop_duplicates()

# Find unique prefixes that the scrubber originates
df2 = df.loc[df['origin_as'] == int(scrubber)]
unique_prefixes = df2["prefix"].unique()

# Convert the numpy array of unique values to a DataFrame
unique_prefixes_df = pd.DataFrame(unique_prefixes, columns=['prefix'])
unique_prefixes_df['time'] = date

# Save the unique values to a CSV file
# unique_prefixes_df.to_csv('../data/as_'+scrubber+'_originated_prefix_'+date+'.csv', index=False)

In [ ]:
# Read csv file and remove duplicate rows
import pandas as pd

scrubber = "13335"
date = "02_jan_2024"
DATE = "2024-01-02" # For update files

path = "../data/02jan2024/"
df = pd.read_csv(path+'as_'+scrubber+'_originated_prefix_'+date+'.csv')

In [ ]:
# Check every 5 minutes BGP updates to see how many new prefixes were announced by Cloudflare
# 1. Find new prefixes originated. NOTE: Code borrowed from github_attack_2018.ipynb.
# check_bgp_updates_step2.py in Aruba

import pybgpstream
import pandas as pd
from datetime import datetime, timedelta
import csv

scrubber = "13335"
date = "02_jan_2024"
DATE = "2024-01-02" # For update files
INTERVAL_MINUTES = 5
path = "../data/"

df = pd.read_csv(path+"/as_"+scrubber+"_originated_prefix_"+date+".csv")
prefixes = df["prefix"]
update_time = []

# Time setup
start_time = datetime.strptime(f"{DATE} 01:00:00", "%Y-%m-%d %H:%M:%S") # 01 is done to make UTC time.
end_time = start_time + timedelta(days=1)
interval = timedelta(minutes=INTERVAL_MINUTES)

current_time = start_time

prefix_all_details = [] # List to store prefixes for throughout a day
while current_time < end_time:
    from_time = int(current_time.timestamp())
    until_time = int((current_time + interval).timestamp())
    stream = pybgpstream.BGPStream(
        from_time = from_time,
        until_time = until_time,
        record_type="updates",
        project="RIS",
        filter = "path _"+scrubber+"$"    
    )

    stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")


    prefix_details = [] # For storing ipv4 prefixes

    # Find paths from DDoS scrubber to route collectors
    for rec in stream.records():
        time = rec.time
        collector = rec.collector
        for elem in rec:

            pfx = elem.fields["prefix"]

            # Find upstream ASN in an AS path
            as_path = elem.fields["as-path"]
            as_path = as_path.split()
            orig = as_path[-1]

            # Store origin asn and announcement time in a dictionary
            asn_time = {}

            # Convert to UTC time
            time_utc = datetime.utcfromtimestamp(time)
#             print("Prefix %s, Origin %s , time %s elem %s" %(pfx, orig, time, elem))

            # Convert UTC datetime to string
            time_utc_str = time_utc.strftime("%Y-%m-%d %H:%M:%S")

            asn_time["time_utc_str"] = time_utc_str

            asn_time["prefix"] = pfx


            prefix_details.append(asn_time)                   

#     print("\n Finding new prefixes announced during that time")
    csv_file = path+'as_'+scrubber+'_originated_prefix_'+date+'_'+ str(until_time) +'.csv' # Output file to store

    df = pd.read_csv(path+'as_'+scrubber+'_originated_prefix_'+date+'.csv')
    prefixes_original = set(df["prefix"]) # Previously presented prefixes

    # Dictionary to hold unique prefixes with their first timestamp
    unique_prefixes = {}

    for entry in prefix_details:
        prefix = entry['prefix']
        timestamp = entry['time_utc_str']
        if prefix not in unique_prefixes and prefix not in prefixes_original: # Store prefix if it is unique and not stored in previous timestamp.
            unique_prefixes[prefix] = timestamp

    # Write to CSV if we find new prefix origniated from the scrubber
    if len(unique_prefixes) != 0:
        with open(csv_file, 'w', newline='') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['prefix', 'time_utc_str'])  # Header
            for prefix, timestamp in unique_prefixes.items():
                writer.writerow([prefix, timestamp])
        print(f"Data has been written to {csv_file} with number of new originated prefixes {len(unique_prefixes)}")
    print("Checked until %s" %until_time)
    current_time += interval # Update time by interval
print("Completed.")

In [ ]:
# Merge each time slots originated prefixes into one
# find_unique_prefix.py in Aruba
import pandas as pd
import csv
import os
import re

path = "/home/shyam/jupy/scrubber_activation/data/02jan2024/"
pattern = r"as_"+scrubber+"_originated_prefix_"+date+"_(\d)+.csv$" 

# List to hold dataframes
dfs = []

for filename in os.listdir(path):
    # Read only processed files
    if re.search(pattern, filename):
#         Read the CSV file and append the dataframe to the list
        df = pd.read_csv(path+filename)
        dfs.append(df)

# Concatenate all DataFrames into a single one
merged_df = pd.concat(dfs, ignore_index=True)
merged_df.to_csv('../data/as_'+scrubber+'_originated_prefix_merged_'+date+'.csv', index=False)
unique_merged_prefixes = merged_df["prefix"].unique()
print("Saved to a file with number of new prefixes on %s are %s and are %s" %(DATE, len(unique_merged_prefixes), unique_merged_prefixes))

In [ ]:
# Check origin of a prefix seen on the previous day with the RIB of the next day.
import pandas as pd
path = "../data/01jan2024/"
scrubber = "13335"
date1 = "01_jan_2024"
date2 = "02_jan_2024"

df = pd.read_csv(path+'as_'+scrubber+'_originated_prefix_merged_'+date1+'.csv')
prefixes = df["prefix"].unique()
prefixes
origin_all = {}
for prefix in prefixes:
    df = pd.read_csv(path + "optimized_raw_as"+scrubber+"_"+date2+".csv")
    origins = df[df["prefix"] == prefix]["origin_as"]
    unique_origins = origins.unique()
    origin_all[prefix] = unique_origins

In [ ]:
df

In [ ]:
# check_origin_asn_next_day.py in aruba
# Check the originator of those prefixes for the next day using RIBs record excep the scrubber
import pybgpstream
import pandas as pd
from datetime import datetime, timedelta

scrubber = "13335"
path = "/data/shared_dir/"

date = "05_jan_2024"
DATE = "2024-01-06" # For update files
start_time = datetime.strptime(f"{DATE} 00:00:00", "%Y-%m-%d %H:%M:%S")

df = pd.read_csv(path + 'as_'+scrubber+'_originated_prefix_merged_'+date+'.csv')
prefixes = df["prefix"].unique()
print("Start time is %s", start_time)
                
for prefix in prefixes:
       
    stream = pybgpstream.BGPStream(
        from_time=str(start_time)+" UTC",
        until_time=str(start_time)+" UTC",
        record_type="ribs",
        project="ris",
        filter = "prefix less "+prefix 
       )
    stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")

    prefix_details = [] # For storing ipv4 prefixes

    print("Starting")
    # Find paths from DDoS scrubber to route collectors
    for rec in stream.records():
        time = rec.time
        collector = rec.collector
        for elem in rec:

            pfx = elem.fields["prefix"]

            # Find upstream ASN in an AS path
            as_path = elem.fields["as-path"]
            as_path = as_path.split()
            orig = as_path[-1]

            if orig != 'scrubber' and pfx != '0.0.0.0/0' and pfx!= '::/0':

                # Store origin asn and announcement time in a dictionary
                asn_time = {}

                asn_time["origin"] = orig

                # Convert to UTC time
#                 print("Prefix %s, Origin %s , time %s elem %s" %(pfx, orig, time, elem))

                asn_time["prefix"] = pfx
#                 prefix_details.append(asn_time)
    print("Here are the details: %s." %asn_time)

In [ ]:
prefix_details

In [ ]:
# Find records that have origin as AS19551 or its siblings.
# Siblings are found using bgp.tools and ASRank API for the date 2019-01-01
siblings = [19905,12008,19911,397215,397228,397219,397221,397233,399158,399167,397243,399166,399169,399157,399155,
            397231,397222,399156,397224,397229,397226,397220,399165,397218,399153,399164,397225,397237,397227,
            397235,399163,399170,399161,399168,397242,399154,397232,397239,397213,397223,397234,399160,397241,
            397238,399159,399152,399177,397240,399171,399179,399162,397216,399176,399180,399173,399172,397236,
            399175,399178,397217,22701,397230,397214,399151,399174]


# Find the number of prefixes of each sibling ASes
for asn in siblings:
    df2 = df.loc[df['origin_as'] == asn]
    unique_prefixes = df2["prefix"].unique()
    print(f"No. of unique prefixes are {len(unique_prefixes)} for ASN {asn}")   


In [ ]:
df_origin_13335 = df.loc[df['origin_as'] == 19551]

a = df_origin_13335["prefix"].unique()
a

In [ ]:
# Check how many of them were suspected to be victim using GRIP API
# Code to call GRIP API for checking MOAS events on a particular date interval for a list of prefixes.

import requests
import json
from datetime import datetime
import urllib.parse
import pandas as pd


prefixes = a

start_time = urllib.parse.quote("2024-01-01T00:00")
end_time = urllib.parse.quote("2024-12-30T00:00")


# Create a defualt dict with all the prefixes as keys and total_suspicious_times = 0.
 # Beause GRIP api returns only prefixs with suspicious events.

suspicious_time_prefix = { }
for prefix in prefixes:
     suspicious_time_prefix[prefix] = [{'total_number_of_events': 0}]

#Iterate over prefixs 
for prefix in prefixes:
    api_url = "https://api.grip.inetintel.cc.gatech.edu/json/events?length=100&start=0&ts_start="+start_time+"&ts_end="+end_time+"&min_susp=80&max_susp=100&event_type=moas&pfxs="+prefix
    response = requests.get(api_url)

    # Check if the request was successful
    if response.status_code == 200:
        # Convert response JSON to a string
        json_string = response.json()
        events = json_string['data']

        suspicious_event_details = [] 
        for i, k in enumerate(events):
                attackers = json_string['data'][i]['summary']['attackers']
                victims = json_string['data'][i]['summary']['victims']
                event_time_unix = int(json_string['data'][i]['view_ts']) 
                event_time = datetime.utcfromtimestamp(event_time_unix).strftime('%Y-%m-%d %H:%M:%S')
                if json_string['data'][i]['finished_ts'] is None:
                    finished_time = "Ongoing"
                else:
                    finished_time_unix = int(json_string['data'][i]['finished_ts'])
                    finished_time = datetime.utcfromtimestamp(finished_time_unix).strftime('%Y-%m-%d %H:%M:%S')

                 # Merge all dictionaries
                merge_all = []

                suspicious_event_counter_dict = { }
                suspicious_event_counter_dict["total_number_of_events"] = len(events) 
                
                # Iterate over data to get start and duration field
                start_duration = { }
                start_duration["start"] = event_time
                start_duration["end"] = finished_time
                start_duration["attacker"] = attackers
                start_duration["victim"] = victims
                
                suspicious_event_details.append(start_duration)
                
                merge_all.append(suspicious_event_counter_dict)
                merge_all.append(suspicious_event_details)
                suspicious_time_prefix[prefix] = merge_all

# Save result into a json file
with open("../data/prefix_moas_"+scrubber+".json", "w") as outfile:
    json.dump(suspicious_time_prefix, outfile)
print(f"Completed.")

In [ ]:
suspicious_time_prefix

In [ ]:
# Check RPKI status of those prefixes using RIPE stat
import requests
import json


scrubber = "32787"
prefixes = a[0:2]

valid_count = 0
invalid_asn_count = 0
invalid_length_count = 0
unknown_count = 0

for prefix in prefixes:
    api_url = "https://stat.ripe.net/data/rpki-validation/data.json?resource="+scrubber+"&prefix="+prefix
    response = requests.get(api_url)
    # response = requests.get(api_url, headers=headers, params=params)
    # Check if the request was successful
    if response.status_code == 200:
        # Convert response JSON to a string
        json_string = response.json()
        roas = json_string['data']['validating_roas']
        print(roas)
        
        results = {}
        results["prefix"] = prefix

        # For each prefix, the API gives different ASNs related to it and their status
        for i, k in enumerate(roas):
            origin_asns = []
#             for r in roas:
                # Find ASNs of differnt lenghts linked to that prefix
#                 status = r[i]['validity']
#                 print(r)
#                 status = r['validity']
#                 origin_asn = r['origin']
#                 if status == "invalid_length":
#                     origin_asns.append(origin_asn)
#                     results["asns"] = origin_asns
                
#                 if status == "valid":
#                     valid_count += 1
#                 elif status == "invalid_asn":
#                     invalid_asn_count += 1
#                 elif status == "invalid_length":
#                     invalid_length_count += 1
#                 else:
#                     unknown_count += 1
print(f"Total: {len(prefixes)}, Valid: #{valid_count}, Invalid ASN {invalid_asn_count}, Invalid length count {invalid_length_count}")

In [2]:
# Read a txt log file produced from the command 
# podman logs check-orgin-asn-next-day-13335-august | grep "Origin is different than 13335" >> filtered-logs.txt

import pandas as pd
import re

# Read the file
with open('../data/filtered-logs.txt', 'r') as file:
    lines = file.readlines()

# Extract origin and prefix from each line
data = []
for line in lines:
    match = re.search(r"'origin':\s*'(\d+)',\s*'prefix':\s*'([\d./]+)'", line)
    if match:
        origin = match.group(1)
        prefix = match.group(2)
        data.append((origin, prefix))

# Create DataFrame and drop duplicates
df = pd.DataFrame(data, columns=['origin', 'prefix']).drop_duplicates()
df.to_csv("../data/as13335_check_origin_asn_23_aug_2024.csv", index=False)
print(df)


        origin            prefix
0       207291     2.57.236.0/24
415     132839   43.251.106.0/23
826     207291   141.136.38.0/24
1241     55532     2.58.107.0/24
1653    132839    45.149.68.0/22
...        ...               ...
736039   23891  203.144.131.0/24
736450   13267   62.240.200.0/21
736862   13267   62.240.192.0/22
737274   13267   62.240.196.0/22
737686   13267   62.240.208.0/20

[1052 rows x 2 columns]


In [3]:
df.to_csv("../data/as13335_check_origin_asn_23_aug_2024.csv", index=False)


In [7]:
# print(len(df["prefix"].unique()))
prefix_counts = df.groupby('origin')['prefix'].count().reset_index(name='prefix_count')
prefix_counts = prefix_counts.sort_values(by='prefix_count', ascending=False)
prefix_counts


,origin,prefix_count
9,132839,137
69,30103,120
20,139646,55
4,11179,53
23,141883,34
...,...,...
44,209029,1
41,205455,1
78,33257,1
39,20473,1


In [8]:
prefix_counts["origin"]

9     132839
69     30103
20    139646
4      11179
23    141883
       ...  
44    209029
41    205455
78     33257
39     20473
67     29970
Name: origin, Length: 135, dtype: object